# Detect objects with the camera

[Back to the introduction](ch01_demo.ipynb). Run in order. This notebook takes a photo with the camera of a laptop, tablet or phone and finds the objects in it with an object-detection model, so the photo never leaves the page. The page must be served over HTTPS (or localhost), and the browser asks for camera permission the first time you tap Start camera.

A notebook kernel runs in a Web Worker, which cannot open the camera. The `CameraCapture` widget therefore shows the camera on the page and hands the kernel a JPEG file (at most 1024 pixels on the longer side), which an image pipeline takes as bytes. Outside JupyterLite, install it with `pip install 'lmlite[camera]'`.

In [ ]:
from lmlite.camera import CameraCapture

camera = CameraCapture()
camera

Tap **Start camera** (pick Rear or Front first on a device with two cameras), aim at something, then tap **Take photo**. The next cell waits for the photo. Tap **Stop camera** when you are done: it releases the camera.

If nothing appears, check that the browser may use the camera for this site, and that no other app is using it. Tap Take photo again, then `camera.clear()` and rerun the cells from `jpeg = await camera.wait()` to detect objects in another photo.

In [ ]:
import io

from PIL import Image as PilImage

jpeg = await camera.wait()
photo = PilImage.open(io.BytesIO(jpeg))
print(f"{photo.width} x {photo.height} pixels, {len(jpeg)} bytes")
photo

## Find the objects

`detr-resnet-50` names the 80 COCO object classes (people, cats, cups, laptops, chairs and so on). It is a model of about 160 MB that runs in the browser on WebAssembly, so a photo takes a few seconds. `threshold` keeps the detections whose score is at least that value: lower it to see more, raise it to see fewer. The boxes are drawn with the built-in bitmap font, because loading a FreeType font hangs in xeus-python.

In [ ]:
from PIL import ImageDraw, ImageFont

from lmlite import pipeline

FONT = ImageFont.load_default_imagefont()
COLORS = ["#e6194b", "#3cb44b", "#4363d8", "#f58231", "#911eb4", "#008080", "#9a6324", "#800000"]


def badge(text, color, scale):
    """White text on a coloured strip, enlarged `scale` times."""
    left, top, right, bottom = FONT.getbbox(text)
    mask = PilImage.new("L", (right, bottom), 0)
    ImageDraw.Draw(mask).text((0, 0), text, font=FONT, fill=255)
    mask = mask.resize((mask.width * scale, mask.height * scale), PilImage.NEAREST)
    strip = PilImage.new("RGB", (mask.width + 8, mask.height + 8), color)
    strip.paste("white", (4, 4), mask)
    return strip


def annotate(image, detections):
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)
    colors = {}
    for item in detections:
        box = item["box"]
        color = colors.setdefault(item["label"], COLORS[len(colors) % len(COLORS)])
        draw.rectangle((box["xmin"], box["ymin"], box["xmax"], box["ymax"]), outline=color, width=4)
        strip = badge(f"{item['label']} {item['score']:.2f}", color, scale=2)
        annotated.paste(strip, (round(box["xmin"]), max(0, round(box["ymin"]) - strip.height)))  # above the box
    return annotated


detector = await pipeline("object-detection", model="detr-resnet-50", backend="wasm")
detections = await detector(jpeg, threshold=0.6)
print(", ".join(f"{item['label']} {item['score']:.2f}" for item in detections) or "No object found: try a lower threshold.")
await detector.dispose()
annotate(photo.convert("RGB"), detections)